<a href="https://colab.research.google.com/github/sebastianczerwinski2/clients_classification/blob/main/clients_classification.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import kagglehub

# Download latest version
path = kagglehub.dataset_download("blastchar/telco-customer-churn")


print("Path to dataset files:", path)

In [ ]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
import matplotlib.pyplot as plt
import seaborn as sns
import glob
import joblib
!pip install catboost
from catboost import CatBoostClassifier

In [ ]:
csv_file = glob.glob(f"{path}/*.csv")
file_to_load = csv_file[0]
data = pd.read_csv(file_to_load)

data.head(n=10)



Zmiana danych na same liczby żeby nasz model mógł na nich pracować

In [ ]:
print(data.dtypes)

data = data.drop(columns='customerID', errors='ignore')
data = data.replace({'Yes': 1, 'No': 0})
data['TotalCharges'] = pd.to_numeric(data['TotalCharges'], errors='coerce')
data['TotalCharges'] = data['TotalCharges'].fillna(0)

data = pd.get_dummies(data,drop_first=True)
data = data.astype(int)
print(data.dtypes)

Trenujemy model regresji liniowej i sprawdzamy podstawowe metryki dla klasyfikacji

In [ ]:
X = data.drop(columns=['Churn'])
y = data['Churn']

X_train, X_test, y_train, y_test = train_test_split(X,y, train_size=0.8, test_size=0.2, random_state=42)

model_base = LogisticRegression()
model_base.fit(X_train,y_train)

predictions_base = model_base.predict(X_test)

acc_base = accuracy_score(y_true=y_test, y_pred=predictions_base)

print(f"Accuracy modelu to: {acc_base} \n")

print(classification_report(y_true=y_test, y_pred=predictions_base))



Nasz model słabo sobie radzi z klientami którzy odchodzą, a są oni najważniejsi bo chcemy ich utrzymać przy sobie, więc zmieniamy parametry modelu żeby skupił się bardziej na klientach którzy potencjalnie chcieliby odejść.

In [ ]:
model_balanced = LogisticRegression(class_weight='balanced')
model_balanced.fit(X_train,y_train)

predictions_balanced = model_balanced.predict(X_test)

acc_balanced = accuracy_score(y_true=y_test, y_pred=predictions_balanced)

print(f"Accuracy modelu to: {acc_balanced} \n")

print(classification_report(y_true=y_test, y_pred=predictions_balanced))


Mamy od razu lepsze wyniki apropo klientów odchodzących (1) szczególnie przy recall, bo zwracamy na nich większą uwagę. Teraz zobaczymy to na macierzach błędu

In [ ]:
cm_base = confusion_matrix(y_test, predictions_base)
cm_balanced = confusion_matrix(y_test, predictions_balanced)

fig, axes = plt.subplots(1,2,figsize=(14,5))

sns.heatmap(cm_base, annot=True, fmt='d', cmap='Blues',
            xticklabels=['Zostaje (0)', 'Odchodzi (1)'],
            yticklabels=['Zostaje (0)', 'Odchodzi (1)'], ax=axes[0])
axes[0].set_title('Zwykły Model (Niezbalansowany)')
axes[0].set_ylabel('Rzeczywistość')
axes[0].set_xlabel('Przewidywania modelu')


sns.heatmap(cm_balanced, annot=True, fmt='d', cmap='Oranges',
            xticklabels=['Zostaje (0)', 'Odchodzi (1)'],
            yticklabels=['Zostaje (0)', 'Odchodzi (1)'], ax=axes[1])
axes[1].set_title('Zbalansowany Model')
axes[1].set_ylabel('Rzeczywistość')
axes[1].set_xlabel('Przewidywania modelu')

plt.tight_layout()
plt.show()

Widzimy że zbalansowane dane pozwalają nam powalczyć aż o 84 klientów więcej kosztem pomyłki apropo klientów którzy zostają, a model pomyślał ze odchodzą i zaproponował im promocję.

In [ ]:
wagi = model_balanced.coef_[0]
nazwy_cech = X.columns

waznosc = pd.DataFrame({
    'Cecha': nazwy_cech,
    'Waga': wagi
})

waznosc = waznosc.sort_values(by='Waga', ascending=False)

top_cechy = pd.concat([waznosc.head(5), waznosc.tail(5)])

plt.figure(figsize=(10,8))

sns.barplot(x='Waga', y='Cecha', data=top_cechy, hue='Cecha')
plt.axvline(x=0, color='black', linestyle='--')

plt.title('Top 10 cech decydujących o odejściu klienta (Regresja Logistyczna)')
plt.xlabel('Wpływ na odejście (Wagi modelu)')
plt.ylabel('Cecha')
plt.tight_layout()
plt.show()


In [ ]:
model_for = RandomForestClassifier(class_weight='balanced' ,random_state=42)

model_for.fit(X_train, y_train)

predictions_forest = model_for.predict(X_test)

acc_forest = accuracy_score(y_test,predictions_forest)
print(f"Accuracy: {acc_forest} \n {classification_report(y_test,predictions_forest)}")

Mimo zbalansowanych danych las losowy słabo sobie radzi więc zmienimy mu próg kiedy będziemy martwić się o odejście klienta

In [ ]:
probability = model_for.predict_proba(X_test)[:,1]
treshold = 0.3
predictions_custom_treshold = (probability >= treshold).astype(int)

print(f"{classification_report(y_test,predictions_custom_treshold)}")

In [ ]:
cm_forest = confusion_matrix(y_test, predictions_custom_treshold)

plt.figure(figsize=(8,6))

sns.heatmap(cm_forest, annot=True, fmt='d', cmap='Greens',
            xticklabels=['Zostaje (0)', 'Odchodzi (1)'],
            yticklabels=['Zostaje (0)', 'Odchodzi (1)'])
plt.title("Macierz pomyłek dla lasu losowego ze zmniejszonym progiem aktywacji")
plt.ylabel("Rzeczywiste dane")
plt.xlabel("Przewidywania")

Widzimy że ten model jest gorszy od zbalansowanej regresji więc sprawdzimy jak poradzi sobie CatBoost

In [ ]:
model_cat = CatBoostClassifier(auto_class_weights='Balanced', random_state=42, verbose=False)

model_cat.fit(X_train, y_train)

predictions_cat = model_cat.predict(X_test)

print(classification_report(y_test, predictions_cat))

In [ ]:
cm_cat = confusion_matrix(y_test, predictions_cat)

plt.figure(figsize=(8,6))

sns.heatmap(cm_cat, annot=True, fmt='d', cmap='Reds',
            xticklabels=['Zostaje (0)', 'Odchodzi (1)'],
            yticklabels=['Zostaje (0)', 'Odchodzi (1)'])
plt.title("Macierz pomyłek dla CatBoost")
plt.ylabel("Rzeczywiste dane")
plt.xlabel("Przewidywania")

Wniosek jaki można wyciągnąć z tych danych jest taki, że czasem najlepsze rozwiazanie jest to najprostsze, ponieważ zbalansowana regresja pokonała swoich bardziej skomplikowanych kolegów. W najważniejszej dla nasz kategorii czyli recall miała najwyższy winik, co możemy też zauważyć na macierzy pomyłek dla kategorii (1)(1) osiągneła najwyższy wynik.

In [ ]:
joblib.dump(model_balanced, 'model_klientow.pkl')